# M2 — Hierarchical Parsing and Standardization

Run this notebook from the code repository root in a CPU-only Colab session after placing the M1 corpus in `data/working`. Copy it into the group's Jupyter Book repository for publication. Every table and chart below is generated during book compilation.

The parser starts at the `\documentclass` file, recursively follows active `\input` and `\include` commands, and ignores unused `.tex` files. It creates Document → heading → paragraph/list → sentence, formula, and figure leaves for every version. Reference sections are excluded; acknowledgements and appendices are retained.


## Standardization and deduplication

Formatting-only commands and float placement hints are removed from each parsed element. Inline math is written as `$...$`; single display blocks use `equation`. Multi-line `align` and `gather` retain their original environment to preserve row alignment and numbering. `\bibitem` and reachable `.bib` entries are extracted into BibTeX fields. Entries with equal DOI, or equal normalized title and compatible author/year, are merged; populated fields are unioned and `\cite` keys rewritten to the selected key. Elements with identical kind and cleaned content share one ID across versions, while hierarchy edges remain version-specific. Empty M1 versions are counted but have no elements.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from src.m2_parsing.pipeline import run_milestone2

working = Path('data/working')
tables = Path('data/tables')
assert working.is_dir(), 'Place the M1 corpus in data/working first'
run_milestone2(working_dir=working, tables_dir=tables)
metrics = json.loads((tables / 'metrics_m2.json').read_text(encoding='utf-8'))
coverage = pd.DataFrame(metrics['coverage'])
pd.DataFrame([{'publications': metrics['n_publications'], 'versions': metrics['n_versions_total'], 'parsed': metrics['n_versions_parsed'], 'empty': metrics['n_versions_empty'], 'success_rate': metrics['version_success_rate']}])


In [ ]:
coverage[['arxiv_id', 'n_versions', 'n_parsed', 'n_empty', 'n_elements_before_dedup', 'n_elements_after_dedup', 'n_bib_before', 'n_bib_after']]


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].bar(['Before', 'After'], [metrics['n_elements_before_dedup'], metrics['n_elements_after_dedup']])
axes[0].set_title('Elements across versions')
axes[1].bar(['Before', 'After'], [metrics['n_bib_before'], metrics['n_bib_after']])
axes[1].set_title('BibTeX entries across versions')
plt.tight_layout()
plt.show()


In [ ]:
pd.DataFrame([{'wall_time_sec': metrics['duration_sec'], 'avg_pub_sec': metrics['avg_publication_sec'], 'peak_ram_mib': metrics['peak_rss_bytes'] / 2**20 if metrics['peak_rss_bytes'] else None, 'output_mib': metrics['output_bytes'] / 2**20}])
